# 台灣金融文件 VectorRAG 統整實作：RAGAS + DeepEval + Arize Phoenix

這份 notebook 是一份「一站式」教學：

1. 使用台灣金融官方文件建立 VectorRAG。
2. 用 GPT 模型回答固定題庫。
3. 以 **RAGAS / DeepEval / Arize Phoenix** 三套工具做完整評測。
4. 統一比較分數並輸出最終結論。

本教學目標是讓新人工程師可以直接照步驟跑出同一套流程。

## 全流程圖

```mermaid
flowchart TD
    A["下載台灣金融官方 PDF"] --> B["切塊 + Embedding 索引"]
    B --> C["Vector Retrieval Top-5"]
    C --> D["GPT 生成答案 (Evidence-only)"]
    D --> E["RAGAS 評測"]
    D --> F["DeepEval 評測"]
    D --> G["Arize Phoenix 評測"]
    E --> H["統一欄位 + 分數比較"]
    F --> H
    G --> H
    H --> I["單題診斷 + 最終結論"]
```


### Cell 1 說明：安裝本 notebook 需要的套件

這一格會安裝三套評測工具與資料處理套件：

- `ragas==0.4.3`
- `deepeval==3.8.4`
- `arize-phoenix==12.33.1` 與 `arize-phoenix-evals==2.9.0`
- `openai`, `pypdf`, `pandas`, `numpy`, `python-dotenv`

若你已在專案 `.venv` 裡安裝完成，可跳過這格。


In [1]:
!uv add ragas==0.4.3 deepeval==3.8.4 arize-phoenix==12.33.1 arize-phoenix-evals==2.9.0 openai pypdf pandas numpy python-dotenv


Resolved 216 packages in 7ms


Audited 214 packages in 28ms


### Cell 2 說明：設定路徑、載入金鑰、下載台灣金融文件

這一格會完成三件事：

1. 設定資料路徑與輸出路徑。
2. 載入 `.env` / `lpdd/.env` 的 `OPENAI_API_KEY`。
3. 從台灣央行官方頁面下載 2025 Financial Stability Report PDF（若本地不存在才下載）。

輸出重點：
- `PDF_PATH`：本次 RAG 的唯一資料來源。
- `openai_client`：後續 embedding 與 LLM 呼叫共用。


In [2]:
import hashlib
import json
import os
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pypdf import PdfReader

PROJECT_ROOT = Path(r"/Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian")
DATA_DIR = PROJECT_ROOT / "docs/Benchmark-Governance/data"
RESULT_DIR = DATA_DIR / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PDF_URL = "https://www.cbc.gov.tw/dl-221034-7e280a7510ec4bbf899d2a177760326f.html"
PDF_PATH = DATA_DIR / "taiwan-financial-stability-report-2025.pdf"
BENCH_PATH = DATA_DIR / "taiwan-finance-rag-benchmark-2025.json"

load_dotenv(PROJECT_ROOT / ".env")
load_dotenv(PROJECT_ROOT / "lpdd/.env")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Please set OPENAI_API_KEY in .env before running this notebook.")

if (not PDF_PATH.exists()) or PDF_PATH.stat().st_size < 1_000_000:
    print(f"Downloading PDF from: {PDF_URL}")
    urllib.request.urlretrieve(PDF_URL, str(PDF_PATH))

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE") or None,
)

print(f"PDF_PATH: {PDF_PATH}")
print(f"PDF_SIZE_MB: {PDF_PATH.stat().st_size / (1024*1024):.2f}")


PDF_PATH: /Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian/docs/Benchmark-Governance/data/taiwan-financial-stability-report-2025.pdf
PDF_SIZE_MB: 5.30


### Cell 3 說明：建立 VectorRAG 檢索索引（text-embedding-3-large）

這一格建立可重現的向量檢索基礎：

1. 解析 PDF 文字。
2. 做基礎清洗與切塊（chunk）。
3. 使用 `text-embedding-3-large` 產生 chunk 向量。
4. 建立 `retrieve_vector(query, top_k=5)`。

效能優化：
- 使用 embedding cache（JSON）避免每次重跑都重做嵌入。

輸出重點：
- `chunks`
- `chunk_embeddings`
- `retrieve_vector(...)`


In [3]:
reader = PdfReader(str(PDF_PATH))
raw_text = "\n".join((p.extract_text() or "") for p in reader.pages)
raw_text = re.sub(r"\s+", " ", raw_text).strip()

chunk_size = 1200
stride = 900
chunks = []
for i in range(0, max(len(raw_text) - chunk_size + 1, 1), stride):
    part = raw_text[i : i + chunk_size].strip()
    if len(part) >= 250:
        chunks.append(part)

EMBED_MODEL = "text-embedding-3-large"
EMBED_CACHE_PATH = RESULT_DIR / f"embedding_cache_taiwan_{EMBED_MODEL}.json"

if EMBED_CACHE_PATH.exists():
    embedding_cache = json.loads(EMBED_CACHE_PATH.read_text(encoding="utf-8"))
else:
    embedding_cache = {}


def _text_key(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _normalize_rows(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.clip(norms, 1e-12, None)


def embed_texts(texts: list[str], batch_size: int = 32) -> np.ndarray:
    keys = [_text_key(t) for t in texts]
    missing = [(k, t) for k, t in zip(keys, texts) if k not in embedding_cache]

    for i in range(0, len(missing), batch_size):
        batch = missing[i : i + batch_size]
        batch_keys = [k for k, _ in batch]
        batch_texts = [t for _, t in batch]

        response = openai_client.embeddings.create(model=EMBED_MODEL, input=batch_texts)
        data_sorted = sorted(response.data, key=lambda x: x.index)
        for k, d in zip(batch_keys, data_sorted):
            embedding_cache[k] = d.embedding

    if missing:
        EMBED_CACHE_PATH.write_text(json.dumps(embedding_cache), encoding="utf-8")

    mat = np.asarray([embedding_cache[k] for k in keys], dtype=np.float32)
    return _normalize_rows(mat)


chunk_embeddings = embed_texts(chunks)


def retrieve_vector(query: str, top_k: int = 5):
    q_vec = embed_texts([query])[0]
    scores = chunk_embeddings @ q_vec
    order = np.argsort(scores)[::-1][:top_k]
    contexts = [chunks[i] for i in order]
    sims = [float(scores[i]) for i in order]
    return contexts, sims


print(f"pages={len(reader.pages)}, chunks={len(chunks)}, cache_entries={len(embedding_cache)}")


pages=123, chunks=298, cache_entries=306


### Cell 4 說明：建立台灣金融題庫（含參考答案）

這裡使用同一份文件建立固定 benchmark 題庫，題目聚焦在：

- 報告目的
- 2024 年總體觀察
- 風險來源
- 市場與機構表現
- 政策因應措施

這份 benchmark 會寫入 JSON，方便後續三套工具共用。


In [4]:
benchmark = [
    {
        "id": "TWFSR-Q01",
        "topic": "report-purpose",
        "question": "台灣央行在金融穩定工作中的核心任務是什麼？和貨幣政策的關係為何？",
        "ground_truth": "央行指出促進金融穩定是其營運目標之一，也是有效執行貨幣政策的基礎；除必要時擔任最終放款者外，亦會定期監測金融體系與總體環境。",
    },
    {
        "id": "TWFSR-Q02",
        "topic": "report-frequency",
        "question": "Financial Stability Report 的發布頻率與主要目的為何？",
        "ground_truth": "該報告為年度發布，目的在揭露對金融穩定的評估、提升透明度與公眾理解，協助及早辨識脆弱性與風險。",
    },
    {
        "id": "TWFSR-Q03",
        "topic": "macro-summary-2024",
        "question": "根據 2025 報告，2024 年台灣金融體系整體狀況如何？",
        "ground_truth": "報告指出 2024 年台灣經濟成長穩健、通膨趨緩，金融市場運作平穩、金融機構表現穩健，整體金融體系維持大致穩定。",
    },
    {
        "id": "TWFSR-Q04",
        "topic": "risk-factors",
        "question": "報告特別提醒需要持續關注的外部不利因素有哪些？",
        "ground_truth": "報告提到需關注美國關稅政策不確定性與氣候變遷等不利因素，並持續監測其對經濟、企業與家庭償債能力的影響。",
    },
    {
        "id": "TWFSR-Q05",
        "topic": "market-and-institutions",
        "question": "報告如何描述 2024 年台灣金融市場與金融機構的表現？",
        "ground_truth": "票債券市場發行與交易量擴增、股市活絡且指數創高、匯市動態穩定；銀行獲利創高且資產品質與資本適足良好，壽險與票券金融公司獲利亦改善。",
    },
    {
        "id": "TWFSR-Q06",
        "topic": "policy-rate",
        "question": "央行在 2024 年 3 月採取了什麼利率政策動作？",
        "ground_truth": "報告指出央行於 2024 年 3 月調升政策利率 0.125 個百分點，並透過公開市場操作調節銀行體系流動性。",
    },
    {
        "id": "TWFSR-Q07",
        "topic": "real-estate-credit",
        "question": "為抑制銀行信用資源過度流向不動產，央行採取了哪些措施？",
        "ground_truth": "央行配合調升存款準備率，兩度調整選擇性信用管制措施，以抑制銀行信用資源過度流向房地產市場。",
    },
    {
        "id": "TWFSR-Q08",
        "topic": "support-plan",
        "question": "面對美國關稅政策衝擊，行政院在 2025 年 4 月提出了什麼支援方案？",
        "ground_truth": "報告指出行政院於 2025 年 4 月提出新台幣 930 億元方案，推動 20 項支持措施協助國內產業因應衝擊。",
    },
]

BENCH_PATH.write_text(json.dumps(benchmark, ensure_ascii=False, indent=2), encoding="utf-8")
benchmark_df = pd.DataFrame(benchmark)
display(benchmark_df)
print(f"Saved benchmark: {BENCH_PATH}")


,id,topic,question,ground_truth
0,TWFSR-Q01,report-purpose,台灣央行在金融穩定工作中的核心任務是什麼？和貨幣政策的關係為何？,央行指出促進金融穩定是其營運目標之一，也是有效執行貨幣政策的基礎；除必要時擔任最終放款者外，...
1,TWFSR-Q02,report-frequency,Financial Stability Report 的發布頻率與主要目的為何？,該報告為年度發布，目的在揭露對金融穩定的評估、提升透明度與公眾理解，協助及早辨識脆弱性與風險。
2,TWFSR-Q03,macro-summary-2024,根據 2025 報告，2024 年台灣金融體系整體狀況如何？,報告指出 2024 年台灣經濟成長穩健、通膨趨緩，金融市場運作平穩、金融機構表現穩健，整體金...
3,TWFSR-Q04,risk-factors,報告特別提醒需要持續關注的外部不利因素有哪些？,報告提到需關注美國關稅政策不確定性與氣候變遷等不利因素，並持續監測其對經濟、企業與家庭償債能...
4,TWFSR-Q05,market-and-institutions,報告如何描述 2024 年台灣金融市場與金融機構的表現？,票債券市場發行與交易量擴增、股市活絡且指數創高、匯市動態穩定；銀行獲利創高且資產品質與資本適...
5,TWFSR-Q06,policy-rate,央行在 2024 年 3 月採取了什麼利率政策動作？,報告指出央行於 2024 年 3 月調升政策利率 0.125 個百分點，並透過公開市場操作調...
6,TWFSR-Q07,real-estate-credit,為抑制銀行信用資源過度流向不動產，央行採取了哪些措施？,央行配合調升存款準備率，兩度調整選擇性信用管制措施，以抑制銀行信用資源過度流向房地產市場。
7,TWFSR-Q08,support-plan,面對美國關稅政策衝擊，行政院在 2025 年 4 月提出了什麼支援方案？,報告指出行政院於 2025 年 4 月提出新台幣 930 億元方案，推動 20 項支持措施協...


Saved benchmark: /Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian/docs/Benchmark-Governance/data/taiwan-finance-rag-benchmark-2025.json


### Cell 5 說明：使用 GPT 模型生成 VectorRAG 回答（Evidence-only）

這一格建立「共用回答輸出」供三套評測工具使用：

1. 先用 `retrieve_vector(..., top_k=5)` 取回證據段落。
2. 使用 `gpt-5-mini` 生成回答。
3. 套用 Evidence-only 規則：
   - 只能依據證據回答
   - 證據不足就回答「我不知道（證據不足）」
   - 可回答時附上證據編號 `[E#]`

輸出重點：
- `rag_df`
- `rows`（後續三套評測直接共用）


In [5]:
def generate_answer_gpt5mini_quote_only(question: str, contexts: list[str], max_tokens: int = 220) -> str:
    evidence_text = "\n\n".join([f"[E{i+1}] {c}" for i, c in enumerate(contexts)])
    prompt = f"""你是金融文件問答助理，只能根據 EVIDENCE 回答。

Question:
{question}

EVIDENCE:
{evidence_text}

規則：
- 只回答問題需要的最小資訊。
- 不可外推，不可補充證據外知識。
- 若證據不足，輸出：我不知道（證據不足）
- 若可回答，句尾必須附上一個 [E#]。
"""

    response = openai_client.chat.completions.create(
        model="gpt-5-mini",
        reasoning_effort="minimal",
        seed=42,
        max_completion_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )

    text = (response.choices[0].message.content or "").strip()
    if not text:
        return "我不知道（證據不足）"
    if "我不知道（證據不足）" not in text and "[E" not in text:
        return "我不知道（證據不足）"
    return text


rows = []
for item in benchmark:
    contexts, retrieval_scores = retrieve_vector(item["question"], top_k=5)
    answer = generate_answer_gpt5mini_quote_only(item["question"], contexts)
    rows.append(
        {
            "id": item["id"],
            "question": item["question"],
            "ground_truth": item["ground_truth"],
            "topic": item["topic"],
            "retrieved_contexts": contexts,
            "retrieval_scores": retrieval_scores,
            "answer": answer,
        }
    )

rag_df = pd.DataFrame(rows)
rag_df.to_csv(RESULT_DIR / "taiwan_vector_rag_answers.csv", index=False)
rag_df.head(3)


,id,question,ground_truth,topic,retrieved_contexts,retrieval_scores,answer
0,TWFSR-Q01,台灣央行在金融穩定工作中的核心任務是什麼？和貨幣政策的關係為何？,央行指出促進金融穩定是其營運目標之一，也是有效執行貨幣政策的基礎；除必要時擔任最終放款者外，...,report-purpose,[velopments in US tariff policies and related ...,"[0.5944929718971252, 0.5673182606697083, 0.566...",我不知道（證據不足）
1,TWFSR-Q02,Financial Stability Report 的發布頻率與主要目的為何？,該報告為年度發布，目的在揭露對金融穩定的評估、提升透明度與公眾理解，協助及早辨識脆弱性與風險。,report-frequency,[he relevant financial authorities and market ...,"[0.6566656231880188, 0.5979018211364746, 0.558...",發布頻率：每年一次（年刊）。主要目的：提供對我國金融體系現況與潛在脆弱性與風險的洞見，並引發...
2,TWFSR-Q03,根據 2025 報告，2024 年台灣金融體系整體狀況如何？,報告指出 2024 年台灣經濟成長穩健、通膨趨緩，金融市場運作平穩、金融機構表現穩健，整體金...,macro-summary-2024,[itate real economic performance in a sustaine...,"[0.6909067034721375, 0.6812684535980225, 0.665...",在 2024 年，台灣的金融體系「整體仍然廣泛穩定」：金融市場運作有序、金融機構獲利改善且資...


### Cell 6 說明：快速人工檢查一題（資料品質與對齊）

在進入自動評測前，先抽一題確認四件事：

1. 問題文字是否正確。
2. 參考答案（ground truth）是否明確。
3. 檢索證據是否與問題相關。
4. GPT 輸出是否遵守 evidence-only 規則。

這一步可快速發現資料前處理或 prompt 設計問題。


In [6]:
target_id = "TWFSR-Q01"
preview_df = rag_df.query("id == @target_id")
display(preview_df[["id", "question", "ground_truth", "answer", "retrieved_contexts"]])


,id,question,ground_truth,answer,retrieved_contexts
0,TWFSR-Q01,台灣央行在金融穩定工作中的核心任務是什麼？和貨幣政策的關係為何？,央行指出促進金融穩定是其營運目標之一，也是有效執行貨幣政策的基礎；除必要時擔任最終放款者外，...,我不知道（證據不足）,[velopments in US tariff policies and related ...


### Cell 7 說明：執行 RAGAS 評測

這一格用 RAGAS 對同一批答案做評估，採用以下指標：

- `faithfulness`
- `answer_relevancy`
- `context_precision`
- `context_recall`
- `context_entity_recall`

注意：
- 目前專案版本為 `ragas==0.4.3`，為了相容 `evaluate(...)` 介面，這裡使用相容寫法。
- 結果會輸出到 `ragas_taiwan_finance_results.csv`。


In [7]:
import warnings
from copy import deepcopy

# Suppress compatibility warnings in ragas==0.4.3 tutorial runs.
warnings.filterwarnings("ignore", category=DeprecationWarning)

from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.embeddings.base import embedding_factory
from ragas.llms import llm_factory
from ragas.metrics import (
    answer_relevancy as m_answer_relevancy,
    context_entity_recall as m_context_entity_recall,
    context_precision as m_context_precision,
    context_recall as m_context_recall,
    faithfulness as m_faithfulness,
)

judge_llm = llm_factory(model="gpt-4o-mini", provider="openai", client=openai_client)
judge_embeddings = embedding_factory("text-embedding-3-small")

faithfulness = deepcopy(m_faithfulness)
faithfulness.llm = judge_llm

answer_relevancy = deepcopy(m_answer_relevancy)
answer_relevancy.llm = judge_llm
answer_relevancy.embeddings = judge_embeddings
answer_relevancy.strictness = 1

context_precision = deepcopy(m_context_precision)
context_precision.llm = judge_llm

context_recall = deepcopy(m_context_recall)
context_recall.llm = judge_llm

context_entity_recall = deepcopy(m_context_entity_recall)
context_entity_recall.llm = judge_llm

samples = [
    SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        retrieved_contexts=r["retrieved_contexts"],
        reference=r["ground_truth"],
    )
    for r in rows
]

dataset = EvaluationDataset(samples=samples)
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    context_entity_recall,
]

ragas_result = evaluate(dataset=dataset, metrics=metrics, show_progress=False)
ragas_df = ragas_result.to_pandas().reset_index(drop=True)

ragas_df["id"] = [r["id"] for r in rows]
ragas_df["tool"] = "ragas"
ragas_df["question"] = [r["question"] for r in rows]
ragas_df["reference"] = [r["ground_truth"] for r in rows]
ragas_df["output"] = [r["answer"] for r in rows]
ragas_df["retrieved_contexts"] = [r["retrieved_contexts"] for r in rows]
ragas_df["hallucination_proxy"] = 1 - pd.to_numeric(ragas_df.get("faithfulness"), errors="coerce")

ragas_csv = RESULT_DIR / "ragas_taiwan_finance_results.csv"
ragas_df.to_csv(ragas_csv, index=False)
print(f"Saved: {ragas_csv}")
ragas_df[["id", "faithfulness", "answer_relevancy", "context_precision", "context_recall", "context_entity_recall"]].head()


/Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved: /Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian/docs/Benchmark-Governance/data/results/ragas_taiwan_finance_results.csv


,id,faithfulness,answer_relevancy,context_precision,context_recall,context_entity_recall
0,TWFSR-Q01,0.0,0.000000,1.000000,1.0,0.000000
1,TWFSR-Q02,1.0,0.439053,0.866667,1.0,0.000000
2,TWFSR-Q03,1.0,0.714298,1.000000,1.0,0.142857
3,TWFSR-Q04,1.0,0.561356,1.000000,1.0,0.000000
4,TWFSR-Q05,1.0,0.700892,1.000000,1.0,0.000000


### Cell 8 說明：執行 DeepEval 評測

這一格使用 DeepEval 做第二套評測。

為了確保在長文件情境下穩定執行，本教學採用較穩定且可重現的三個指標：

- `AnswerRelevancyMetric`
- `ContextualPrecisionMetric`
- `GEval(Financial Groundedness)`

註記：
- 若直接啟用所有高成本指標（特別是長上下文 Faithfulness / Hallucination）可能觸發 token 長度上限。
- 這裡先完成可穩定落地的核心評測，再做跨工具比較。

輸出檔：
- `deepeval_taiwan_finance_results.csv`


In [8]:
from deepeval import evaluate as deepeval_evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig
from deepeval.metrics import AnswerRelevancyMetric, ContextualPrecisionMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Avoid per-attempt timeout on long context prompts.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "180"


test_cases = []
for r in rows:
    compact_context = [c[:700] for c in r["retrieved_contexts"][:3]]
    test_cases.append(
        LLMTestCase(
            input=r["question"],
            actual_output=r["answer"],
            expected_output=r["ground_truth"],
            retrieval_context=compact_context,
            context=compact_context,
            name=r["id"],
        )
    )

metrics = [
    AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini", async_mode=False),
    ContextualPrecisionMetric(threshold=0.7, model="gpt-4o-mini", async_mode=False),
    GEval(
        name="Financial Groundedness",
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ],
        criteria="Answer should be financially conservative, evidence-grounded, and avoid unsupported claims.",
        threshold=0.75,
        model="gpt-4o-mini",
        async_mode=False,
    ),
]

deep_eval_result = deepeval_evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=AsyncConfig(run_async=False),
    display_config=DisplayConfig(show_indicator=False, print_results=True),
)

metric_name_map = {
    "Answer Relevancy": "answer_relevancy",
    "Contextual Precision": "contextual_precision",
    "Financial Groundedness [GEval]": "financial_groundedness",
    "Financial Groundedness": "financial_groundedness",
}

records = []
for tr in deep_eval_result.test_results:
    row = {
        "tool": "deepeval",
        "id": tr.name,
        "question": tr.input,
        "reference": tr.expected_output,
        "output": tr.actual_output,
        "retrieved_contexts": tr.retrieval_context,
    }
    for md_item in tr.metrics_data:
        metric_key = metric_name_map.get(md_item.name, md_item.name.lower().replace(" ", "_"))
        row[f"{metric_key}_score"] = md_item.score
        row[f"{metric_key}_reason"] = md_item.reason
        row[f"{metric_key}_success"] = md_item.success
    records.append(row)

deepeval_df = pd.DataFrame(records)

grounded = pd.to_numeric(deepeval_df.get("financial_groundedness_score"), errors="coerce")
deepeval_df["faithfulness_score_proxy"] = grounded
deepeval_df["hallucination_score_proxy"] = 1 - grounded

deepeval_csv = RESULT_DIR / "deepeval_taiwan_finance_results.csv"
deepeval_df.to_csv(deepeval_csv, index=False)
print(f"Saved: {deepeval_csv}")

display_cols = [
    c
    for c in [
        "id",
        "answer_relevancy_score",
        "contextual_precision_score",
        "financial_groundedness_score",
        "hallucination_score_proxy",
    ]
    if c in deepeval_df.columns
]

deepeval_df[display_cols].head()




Metrics Summary

  - ❌ Answer Relevancy (score: 0.5, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.50 because the output included an irrelevant statement about '證據不足' which does not directly relate to the core tasks of the central bank or its relationship with monetary policy. This detracted from the overall relevance, but the response still contained some pertinent information., error: None)
  - ✅ Contextual Precision (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because all relevant nodes are ranked higher than the irrelevant node. The first node provides a clear explanation that 'Promoting financial stability not only is one of the operational objectives pursued by the Central Bank of the Republic of China (Taiwan), but also lays the cornerstone for the effective implementation of monetary policy,' directly addressing the core mission of the central bank. The second node further supports 

⚠ WARNING: No hyperparameters logged.
» ]8;id=882005;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 86.87s | token cost: 0.00736635 USD)
» Test Results (8 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 8

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Saved: /Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian/docs/Benchmark-Governance/data/results/deepeval_taiwan_finance_results.csv


,id,answer_relevancy_score,contextual_precision_score,financial_groundedness_score,hallucination_score_proxy
0,TWFSR-Q01,0.5,1.0,0.003733,0.996267
1,TWFSR-Q02,1.0,1.0,0.704552,0.295448
2,TWFSR-Q03,1.0,1.0,0.715282,0.284718
3,TWFSR-Q04,1.0,1.0,0.661501,0.338499
4,TWFSR-Q05,1.0,1.0,0.650000,0.350000


### Cell 9 說明：執行 Arize Phoenix（Modern API）評測

這一格使用 Phoenix 新版 evaluator 介面：

- `FaithfulnessEvaluator`
- `CorrectnessEvaluator`
- `DocumentRelevanceEvaluator`

並把 Phoenix score 物件展平為：
- `*_score`
- `*_label`
- `*_explanation`

同時建立相容欄位：
- `qa_score = correctness_score`
- `relevance_score = document_relevance_score`
- `hallucination_score = 1 - faithfulness_score`

輸出檔：
- `phoenix_taiwan_finance_results.csv`


In [9]:
from phoenix.evals import LLM, create_evaluator, evaluate_dataframe
from phoenix.evals.metrics import CorrectnessEvaluator, DocumentRelevanceEvaluator, FaithfulnessEvaluator


def _join_contexts(value) -> str:
    if isinstance(value, list):
        return "\n\n".join(str(v) for v in value if str(v).strip())
    if value is None:
        return ""
    return str(value)


base_df = rag_df[["id", "question", "ground_truth", "answer", "retrieved_contexts"]].rename(
    columns={"question": "input", "ground_truth": "reference", "answer": "output"}
)
base_df["context"] = base_df["retrieved_contexts"].apply(_join_contexts)
base_df["document_text"] = base_df["retrieved_contexts"].apply(_join_contexts)


@create_evaluator(name="answer_length", kind="code", direction="neutral")
def answer_length(output: str) -> float:
    return float(len(output))


@create_evaluator(name="contains_finance_terms", kind="code", direction="maximize")
def contains_finance_terms(output: str) -> bool:
    terms = ["financial", "risk", "stability", "bank", "credit", "inflation"]
    out = output.lower()
    return any(t in out for t in terms)


def _score_to_parts(value):
    if isinstance(value, dict):
        return {
            "numeric": value.get("score"),
            "label": value.get("label"),
            "explanation": value.get("explanation"),
            "direction": value.get("direction"),
            "raw": value,
        }
    if isinstance(value, (int, float, np.number, bool)):
        return {"numeric": float(value), "label": None, "explanation": None, "direction": None, "raw": value}
    return {"numeric": np.nan, "label": None, "explanation": None, "direction": None, "raw": value}


def _flatten_metric_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    metric_cols = [c for c in out.columns if c.endswith("_score")]
    for col in metric_cols:
        parts = out[col].apply(_score_to_parts).apply(pd.Series)
        metric_name = col[: -len("_score")]
        out[col] = pd.to_numeric(parts["numeric"], errors="coerce")
        out[f"{metric_name}_raw"] = parts["raw"]
        if parts["label"].notna().any():
            out[f"{metric_name}_label"] = parts["label"]
        if parts["explanation"].notna().any():
            out[f"{metric_name}_explanation"] = parts["explanation"]
    return out


judge_llm = LLM(
    provider="openai",
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE") or None,
)

phoenix_raw = evaluate_dataframe(
    dataframe=base_df.copy(),
    evaluators=[
        answer_length,
        contains_finance_terms,
        FaithfulnessEvaluator(judge_llm),
        CorrectnessEvaluator(judge_llm),
        DocumentRelevanceEvaluator(judge_llm),
    ],
    hide_tqdm_bar=False,
)

phoenix_df = _flatten_metric_columns(phoenix_raw)
phoenix_df.insert(0, "tool", "phoenix")

phoenix_df["qa_score"] = pd.to_numeric(phoenix_df.get("correctness_score"), errors="coerce")
phoenix_df["qa_label"] = phoenix_df.get("correctness_label")
phoenix_df["qa_explanation"] = phoenix_df.get("correctness_explanation")

phoenix_df["relevance_score"] = pd.to_numeric(phoenix_df.get("document_relevance_score"), errors="coerce")
phoenix_df["relevance_label"] = phoenix_df.get("document_relevance_label")
phoenix_df["relevance_explanation"] = phoenix_df.get("document_relevance_explanation")

faithfulness_score = pd.to_numeric(phoenix_df.get("faithfulness_score"), errors="coerce")
phoenix_df["hallucination_score"] = 1 - faithfulness_score
phoenix_df["hallucination_label"] = phoenix_df.get("faithfulness_label").map({"faithful": "factual", "unfaithful": "hallucinated"})
phoenix_df["hallucination_explanation"] = phoenix_df.get("faithfulness_explanation")
phoenix_df["hallucination_quality"] = faithfulness_score

phoenix_csv = RESULT_DIR / "phoenix_taiwan_finance_results.csv"
phoenix_df.to_csv(phoenix_csv, index=False)
print(f"Saved: {phoenix_csv}")
phoenix_df[["id", "faithfulness_score", "correctness_score", "document_relevance_score", "hallucination_score"]].head()


Evaluating Dataframe |          | 0/40 (0.0%) | ⏳ 00:00<? | ?it/s

Evaluating Dataframe |▊         | 3/40 (7.5%) | ⏳ 00:01<00:15 |  2.41it/s

Evaluating Dataframe |█         | 4/40 (10.0%) | ⏳ 00:02<00:25 |  1.41it/s

Evaluating Dataframe |█▎        | 5/40 (12.5%) | ⏳ 00:03<00:28 |  1.22it/s

Evaluating Dataframe |██        | 8/40 (20.0%) | ⏳ 00:04<00:16 |  1.95it/s

...

Evaluating Dataframe |█████████▊| 39/40 (97.5%) | ⏳ 00:27<00:00 |  1.23it/s

Evaluating Dataframe |██████████| 40/40 (100.0%) | ⏳ 00:28<00:00 |  1.16it/s

Evaluating Dataframe |██████████| 40/40 (100.0%) | ⏳ 00:28<00:00 |  1.41it/s

Saved: /Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian/docs/Benchmark-Governance/data/results/phoenix_taiwan_finance_results.csv


,id,faithfulness_score,correctness_score,document_relevance_score,hallucination_score
0,TWFSR-Q01,0.0,0.0,1.0,1.0
1,TWFSR-Q02,1.0,1.0,1.0,0.0
2,TWFSR-Q03,1.0,1.0,1.0,0.0
3,TWFSR-Q04,1.0,1.0,1.0,0.0
4,TWFSR-Q05,1.0,1.0,1.0,0.0


### Cell 10 說明：統一三套工具分數格式並做橫向比較

不同工具的欄位命名不同，這一格會先做欄位對齊，再產出：

1. `summary_df`：每套工具的平均分。
2. `per_question_compare`：每題在三工具下的核心分數。

本格採用的統一欄位語意：

- `groundedness`：回答是否忠於證據（越高越好）
- `answer_quality`：是否正確且切題（越高越好）
- `retrieval_relevance`：檢索內容是否對問題有用（越高越好）
- `hallucination`：幻覺程度（越低越好）


In [10]:
def _safe_mean(series: pd.Series):
    numeric = pd.to_numeric(series, errors="coerce")
    return float(numeric.mean()) if numeric.notna().any() else float("nan")


ragas_std = pd.DataFrame(
    {
        "tool": "ragas",
        "id": ragas_df["id"],
        "question": ragas_df["question"],
        "groundedness": pd.to_numeric(ragas_df.get("faithfulness"), errors="coerce"),
        "answer_quality": pd.to_numeric(ragas_df.get("answer_relevancy"), errors="coerce"),
        "retrieval_relevance": pd.to_numeric(ragas_df.get("context_precision"), errors="coerce"),
        "hallucination": 1 - pd.to_numeric(ragas_df.get("faithfulness"), errors="coerce"),
    }
)

deepeval_grounded = pd.to_numeric(
    deepeval_df.get("faithfulness_score", deepeval_df.get("financial_groundedness_score")),
    errors="coerce",
)
deepeval_hallu = pd.to_numeric(
    deepeval_df.get("hallucination_score", deepeval_df.get("hallucination_score_proxy", 1 - deepeval_grounded)),
    errors="coerce",
)

deepeval_std = pd.DataFrame(
    {
        "tool": "deepeval",
        "id": deepeval_df["id"],
        "question": deepeval_df["question"],
        "groundedness": deepeval_grounded,
        "answer_quality": pd.to_numeric(deepeval_df.get("answer_relevancy_score"), errors="coerce"),
        "retrieval_relevance": pd.to_numeric(deepeval_df.get("contextual_precision_score"), errors="coerce"),
        "hallucination": deepeval_hallu,
    }
)

phoenix_std = pd.DataFrame(
    {
        "tool": "phoenix",
        "id": phoenix_df["id"],
        "question": phoenix_df["input"],
        "groundedness": pd.to_numeric(phoenix_df.get("faithfulness_score"), errors="coerce"),
        "answer_quality": pd.to_numeric(phoenix_df.get("correctness_score"), errors="coerce"),
        "retrieval_relevance": pd.to_numeric(phoenix_df.get("document_relevance_score"), errors="coerce"),
        "hallucination": pd.to_numeric(phoenix_df.get("hallucination_score"), errors="coerce"),
    }
)

all_std = pd.concat([ragas_std, deepeval_std, phoenix_std], ignore_index=True)

summary_df = (
    all_std.groupby("tool", as_index=False)
    .agg(
        rows=("id", "count"),
        groundedness=("groundedness", "mean"),
        answer_quality=("answer_quality", "mean"),
        retrieval_relevance=("retrieval_relevance", "mean"),
        hallucination=("hallucination", "mean"),
    )
    .sort_values("tool")
)

per_question_compare = (
    all_std.pivot_table(
        index=["id", "question"],
        columns="tool",
        values=["groundedness", "answer_quality", "retrieval_relevance", "hallucination"],
        aggfunc="mean",
    )
    .reset_index()
)

summary_csv = RESULT_DIR / "taiwan_cross_tool_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("Saved:", summary_csv)
print("\n=== Summary Metrics ===")
display(summary_df)
print("\n=== Per-Question Compare ===")
display(per_question_compare)


Saved: /Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian/docs/Benchmark-Governance/data/results/taiwan_cross_tool_summary.csv

=== Summary Metrics ===


,tool,rows,groundedness,answer_quality,retrieval_relevance,hallucination
0,deepeval,8,0.592768,0.937500,0.937500,0.407232
1,phoenix,8,0.875000,0.750000,1.000000,0.125000
2,ragas,8,0.875000,0.529355,0.983333,0.125000



=== Per-Question Compare ===


id                                  question answer_quality  \
tool                                                            deepeval   
0     TWFSR-Q01          台灣央行在金融穩定工作中的核心任務是什麼？和貨幣政策的關係為何？            0.5   
1     TWFSR-Q02  Financial Stability Report 的發布頻率與主要目的為何？            1.0   
2     TWFSR-Q03            根據 2025 報告，2024 年台灣金融體系整體狀況如何？            1.0   
3     TWFSR-Q04                   報告特別提醒需要持續關注的外部不利因素有哪些？            1.0   
4     TWFSR-Q05              報告如何描述 2024 年台灣金融市場與金融機構的表現？            1.0   
5     TWFSR-Q06                央行在 2024 年 3 月採取了什麼利率政策動作？            1.0   
6     TWFSR-Q07               為抑制銀行信用資源過度流向不動產，央行採取了哪些措施？            1.0   
7     TWFSR-Q08      面對美國關稅政策衝擊，行政院在 2025 年 4 月提出了什麼支援方案？            1.0   

                       groundedness               hallucination                \
tool phoenix     ragas     deepeval phoenix ragas      deepeval phoenix ragas   
0        0.0  0.000000     0.003733     0.0   0.0      0.996267     1.0   1.0   
1        1.0  0.439053     0.704552     1.0   1.0      0.295448     0.0   0.0   
2        1.0  0.714298     0.715282     1.0   1.0      0.284718     0.0   0.0   
3        1.0  0.561356     0.661501     1.0   1.0      0.338499     0.0   0.0   
4        1.0  0.700892     0.650000     1.0   1.0      0.350000     0.0   0.0   
5        0.0  0.774995     0.642107     1.0   1.0      0.357893     0.0   0.0   
6        1.0  0.605277     0.717751     1.0   1.0      0.282249     0.0   0.0   
7        1.0  0.438968     0.647220     1.0   1.0      0.352780     0.0   0.0   

     retrieval_relevance                    
tool            deepeval phoenix     ragas  
0                    1.0     1.0  1.000000  
1                    1.0     1.0  0.866667  
2                    1.0     1.0  1.000000  
3                    1.0     1.0  1.000000  
4                    1.0     1.0  1.000000  
5                    1.0     1.0  1.000000  
6                    0.5     1.0  1.000000  
7                    1.0     1.0  1.000000

### Cell 11 說明：單一問題跨工具解析（分數如何對應）

這一格以單一題目為例，對照三套工具結果，理解：

- 哪一套工具覺得答案比較穩健
- 哪一套工具對回答正確性更嚴格
- 檢索相關度與幻覺分數是否一致

建議閱讀順序：
1. 先看 `groundedness`
2. 再看 `answer_quality`
3. 最後對照 `hallucination`


In [11]:
focus_id = "TWFSR-Q01"
focus_df = all_std.query("id == @focus_id").copy()
focus_df = focus_df[["tool", "id", "question", "groundedness", "answer_quality", "retrieval_relevance", "hallucination"]]
display(focus_df)

if not focus_df.empty:
    notes = []
    for _, row in focus_df.iterrows():
        notes.append(
            {
                "tool": row["tool"],
                "interpretation": (
                    f"groundedness={row['groundedness']:.3f}, "
                    f"answer_quality={row['answer_quality']:.3f}, "
                    f"retrieval_relevance={row['retrieval_relevance']:.3f}, "
                    f"hallucination={row['hallucination']:.3f}"
                ),
            }
        )
    display(pd.DataFrame(notes))


,tool,id,question,groundedness,answer_quality,retrieval_relevance,hallucination
0,ragas,TWFSR-Q01,台灣央行在金融穩定工作中的核心任務是什麼？和貨幣政策的關係為何？,0.000000,0.0,1.0,1.000000
8,deepeval,TWFSR-Q01,台灣央行在金融穩定工作中的核心任務是什麼？和貨幣政策的關係為何？,0.003733,0.5,1.0,0.996267
16,phoenix,TWFSR-Q01,台灣央行在金融穩定工作中的核心任務是什麼？和貨幣政策的關係為何？,0.000000,0.0,1.0,1.000000


,tool,interpretation
0,ragas,"groundedness=0.000, answer_quality=0.000, retr..."
1,deepeval,"groundedness=0.004, answer_quality=0.500, retr..."
2,phoenix,"groundedness=0.000, answer_quality=0.000, retr..."


### Cell 12 說明：輸出最終結論（Markdown）

這一格會把本次結果整理成可直接貼到文件或 PR 的結論段。

內容包含：

1. 三工具平均分數摘要。
2. 整體觀察（優勢與瓶頸）。
3. 下一輪改善優先順序。


In [12]:
summary_lookup = summary_df.set_index("tool").to_dict(orient="index")


def _fmt(v):
    if pd.isna(v):
        return "NA"
    return f"{float(v):.3f}"

lines = []
lines.append("## 最終結論\n")
lines.append("| Tool | Groundedness(越高越好) | Answer Quality(越高越好) | Retrieval Relevance(越高越好) | Hallucination(越低越好) |")
lines.append("|---|---:|---:|---:|---:|")
for tool in ["ragas", "deepeval", "phoenix"]:
    row = summary_lookup.get(tool, {})
    lines.append(
        f"| {tool} | {_fmt(row.get('groundedness'))} | {_fmt(row.get('answer_quality'))} | {_fmt(row.get('retrieval_relevance'))} | {_fmt(row.get('hallucination'))} |"
    )

lines.append("\n### 觀察重點")
lines.append("1. 三套工具都能在同一份 VectorRAG 輸出上完成評測，流程可重現。")
lines.append("2. 若 `retrieval_relevance` 高但 `answer_quality` 仍低，通常是生成階段未精準對齊問題。")
lines.append("3. 若 `groundedness` 低且 `hallucination` 高，優先收斂『只引用 evidence、證據不足回答不知道』規則。")

lines.append("\n### 下一步建議")
lines.append("1. 針對最低分題目優先調整檢索（chunk 策略、top-k、rerank）。")
lines.append("2. 在生成後加入 answer-evidence 對齊檢查，不符合就回覆證據不足。")
lines.append("3. 每次只改一個維度，持續用這三套工具做回歸比較。")

final_md = "\n".join(lines)
display(Markdown(final_md))


## 最終結論

| Tool | Groundedness(越高越好) | Answer Quality(越高越好) | Retrieval Relevance(越高越好) | Hallucination(越低越好) |
|---|---:|---:|---:|---:|
| ragas | 0.875 | 0.529 | 0.983 | 0.125 |
| deepeval | 0.593 | 0.938 | 0.938 | 0.407 |
| phoenix | 0.875 | 0.750 | 1.000 | 0.125 |

### 觀察重點
1. 三套工具都能在同一份 VectorRAG 輸出上完成評測，流程可重現。
2. 若 `retrieval_relevance` 高但 `answer_quality` 仍低，通常是生成階段未精準對齊問題。
3. 若 `groundedness` 低且 `hallucination` 高，優先收斂『只引用 evidence、證據不足回答不知道』規則。

### 下一步建議
1. 針對最低分題目優先調整檢索（chunk 策略、top-k、rerank）。
2. 在生成後加入 answer-evidence 對齊檢查，不符合就回覆證據不足。
3. 每次只改一個維度，持續用這三套工具做回歸比較。

### Cell 13 結語

你現在已完成完整流程：

- 同一份台灣金融官方文件
- 同一套 VectorRAG
- 同一批答案
- 三套評測工具橫向比較

這份 notebook 可直接作為團隊 onboarding 與回歸測試模板。
